# From Voice to Text and Back: ASR and TTS

This notebook analyzes a speech sample, visualizes its waveform and log-Mel spectrum, compares Wav2Vec2/CTC with Whisper, calculates WER, and synthesizes speech from a transcript.

**Important:** the transcript in the WER cell is only a provisional reference copied from the project. Verify or replace it using a human transcript checked against the audio before treating WER as final accuracy. The models are demonstrations, not stutter-diagnosis tools.

In Google Colab, run the setup cell and upload the audio when prompted. Locally, the notebook loads `audio/new_stutter.mp3` from the repository.

In [ ]:
%pip install librosa matplotlib jiwer faster-whisper gTTS transformers

In [ ]:
from pathlib import Path
import sys

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

AUDIO_PATH = Path("audio/new_stutter.mp3")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not AUDIO_PATH.is_file():
    if "google.colab" in sys.modules:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("Upload an audio file to continue.")
        AUDIO_PATH = Path(next(iter(uploaded)))
    else:
        raise FileNotFoundError(
            f"Audio file not found: {AUDIO_PATH}. Set AUDIO_PATH to your recording."
        )

audio, source_sr = librosa.load(AUDIO_PATH, sr=None, mono=True)
audio_16k = librosa.resample(audio, orig_sr=source_sr, target_sr=16000)
duration = len(audio) / source_sr
print(f"Audio: {AUDIO_PATH} | {source_sr} Hz | {duration:.2f} seconds")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
librosa.display.waveshow(audio, sr=source_sr, ax=axes[0])
axes[0].set(title="Speech waveform", xlabel="Time (s)", ylabel="Amplitude")

mel = librosa.feature.melspectrogram(
    y=audio, sr=source_sr, n_fft=2048, hop_length=512, n_mels=128
)
log_mel = librosa.power_to_db(mel, ref=np.max)
image = librosa.display.specshow(
    log_mel, sr=source_sr, hop_length=512,
    x_axis="time", y_axis="mel", ax=axes[1]
)
axes[1].set(title="Log-Mel spectrogram", xlabel="Time (s)", ylabel="Mel frequency")
fig.colorbar(image, ax=axes[1], format="%+2.0f dB")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "notebook_waveform_spectrogram.png", dpi=180)
plt.show()
print(f"Log-Mel feature shape: {log_mel.shape}")

## Wav2Vec2 ASR with CTC

Wav2Vec2 consumes normalized waveform samples. Its CTC head predicts token scores at many audio frames; the processor's decoder collapses repeats and removes blank tokens.

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

wav2vec_model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(wav2vec_model_name)
wav2vec_model = Wav2Vec2ForCTC.from_pretrained(wav2vec_model_name)
inputs = processor(audio_16k, sampling_rate=16000, return_tensors="pt")
with torch.inference_mode():
    logits = wav2vec_model(inputs.input_values).logits
wav2vec_text = processor.batch_decode(logits.argmax(dim=-1))[0]
print("Wav2Vec2 transcript:")
print(wav2vec_text)

## Whisper ASR

Whisper is a separate encoder-decoder Transformer system. It uses log-Mel audio features and generates text tokens autoregressively; it does **not** use CTC.

In [ ]:
import av

# Compatibility for PyAV builds whose av.open no longer accepts this keyword.
if not getattr(av.open, "_accepts_metadata_errors", False):
    _av_open = av.open
    def _open_compat(*args, **kwargs):
        kwargs.pop("metadata_errors", None)
        return _av_open(*args, **kwargs)
    _open_compat._accepts_metadata_errors = True
    av.open = _open_compat

from faster_whisper import WhisperModel

whisper_model = WhisperModel("base", device="cpu", compute_type="int8")
segments, whisper_info = whisper_model.transcribe(
    str(AUDIO_PATH), language="en", beam_size=5
)
whisper_parts = [segment.text.strip() for segment in segments]
whisper_text = " ".join(part for part in whisper_parts if part).strip()
print(f"Whisper language: {whisper_info.language} ({whisper_info.language_probability:.2f})")
print("Whisper transcript:")
print(whisper_text)

## CTC decoding concept

CTC (Connectionist Temporal Classification) permits many frame-to-token alignments for one transcript. Greedy decoding takes the most likely token per frame, first collapses adjacent repeated tokens, and then removes blank tokens. The order matters: `A, blank, A` becomes `AA`, while `A, A` becomes `A`.

In [ ]:
frame_tokens = [0, 1, 1, 0, 1, 2, 2, 0]
blank_id = 0
collapsed = [token for index, token in enumerate(frame_tokens)
             if index == 0 or token != frame_tokens[index - 1]]
decoded = [token for token in collapsed if token != blank_id]
print("Frame tokens (0 is blank):", frame_tokens)
print("Collapse adjacent repeats:", collapsed)
print("Remove blanks:", decoded, "=> AAB")

## WER comparison

Edit the reference below to a human-verified verbatim transcript. The current text mirrors the project's provisional reference; because it was not independently verified, the calculated metrics are illustrative only. WER = (substitutions + deletions + insertions) / reference word count.

In [ ]:
from jiwer import process_words, wer

reference = input("Paste a human-verified reference transcript: ").strip()
if not reference:
    raise ValueError("Enter a verified reference transcript before calculating WER.")
hypotheses = {"Wav2Vec2 CTC": wav2vec_text, "Whisper": whisper_text}
wer_rows = []
for model_name, hypothesis in hypotheses.items():
    details = process_words(reference, hypothesis)
    score = wer(reference, hypothesis)
    row = {
        "model": model_name,
        "reference_words": len(reference.split()),
        "hits": details.hits,
        "substitutions": details.substitutions,
        "deletions": details.deletions,
        "insertions": details.insertions,
        "wer": score,
    }
    wer_rows.append(row)
    print(row)

import pandas as pd
wer_table = pd.DataFrame(wer_rows)
display(wer_table)
wer_table.to_csv(OUTPUT_DIR / "notebook_wer_results.csv", index=False)

## Text to speech

This cell uses Google Text-to-Speech and therefore needs an internet connection. It synthesizes the Whisper transcript, saves an MP3, and plays it in the notebook. It does not recreate the original speaker or stutter.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

tts_path = OUTPUT_DIR / "notebook_tts_output.mp3"
gTTS(text=whisper_text, lang="en").save(str(tts_path))
print(f"TTS audio saved to: {tts_path}")
display(Audio(filename=str(tts_path)))

## Interpretation and limitations

The waveform and spectrogram visualize acoustic structure; they do not by themselves diagnose a speech disorder. ASR outputs can differ in disfluencies, punctuation, and word choice. The comparison uses one short recording and a provisional reference, so it is not a general benchmark. Replace the reference with a transcript verified by listening, then rerun WER before drawing accuracy conclusions. See `report/asr_architecture.svg` and `report/three_minute_demo.md` for the architecture and presentation materials.